# **Day 54: Iterators**

We have written `for x in something` many times. Today we learn what actually happens **behind** a for loop.

- **Iterable:** Anything we can loop over (list, string, dictionary, file). It has the `__iter__()` method.
- **Iterator:** The object that actually **gives the next value**, one at a time. It has the `__next__()` method. When there are no more values, it raises `StopIteration`.

**Simple example:** A **book** is an iterable. A **bookmark** is the iterator: it remembers which page we are on and gives us the next page.

In [1]:
stations = ["Howrah", "Liluah", "Belur", "Bally"]

it = iter(stations)        # Get an iterator from the list
print(type(it))
print(next(it))
print(next(it))
print(next(it))
print(next(it))
# print(next(it))          # This will raise StopIteration (no more stations)

<class 'list_iterator'>
Howrah
Liluah
Belur
Bally


## **What a for Loop Really Does**
A `for` loop is just a short form of this code:

In [2]:
it = iter(stations)
while True:
    try:
        s = next(it)
    except StopIteration:
        break
    print(s)

Howrah
Liluah
Belur
Bally


## **An Iterable Can Be Used Many Times, an Iterator Only Once**

In [3]:
nums = [1, 2, 3]
print(list(nums), list(nums))       # A list works again and again

it = iter(nums)
print(list(it), list(it))           # An iterator gets used up

[1, 2, 3] [1, 2, 3]
[1, 2, 3] []


## **Making Our Own Iterator**
A class becomes an iterator if it has:
- `__iter__()`: returns the iterator object (usually `self`)
- `__next__()`: returns the next value, or raises `StopIteration` at the end

In [4]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1

for n in Countdown(5):
    print(n, end=" ")

5 4 3 2 1 

## **Example from Remote Sensing: Reading a Big Image in Tiles**
A satellite image can be very large (for example, 10,980 x 10,980 pixels for one Sentinel-2 band). Loading the whole image at once can fill up the computer's memory.

So GIS software reads the image in small square pieces called **tiles** (or **windows**), one at a time. Let us make an iterator that gives the position of each tile.

In [5]:
class TileIterator:
    """Gives (row_start, row_end, col_start, col_end) for each tile of an image."""

    def __init__(self, height, width, tile_size):
        self.height = height
        self.width = width
        self.tile_size = tile_size
        self.row = 0
        self.col = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.row >= self.height:
            raise StopIteration
        r0, c0 = self.row, self.col
        r1 = min(r0 + self.tile_size, self.height)
        c1 = min(c0 + self.tile_size, self.width)
        # Move to the next tile: right first, then down to the next row of tiles
        self.col += self.tile_size
        if self.col >= self.width:
            self.col = 0
            self.row += self.tile_size
        return r0, r1, c0, c1

# A small image of 1000 x 1500 pixels, read in tiles of 512 x 512
tiles = list(TileIterator(1000, 1500, 512))
print("Number of tiles:", len(tiles))
for t in tiles:
    r0, r1, c0, c1 = t
    print(f"rows {r0:>4}-{r1:<5} cols {c0:>4}-{c1:<5} size {r1 - r0} x {c1 - c0}")

Number of tiles: 6
rows    0-512   cols    0-512   size 512 x 512
rows    0-512   cols  512-1024  size 512 x 512
rows    0-512   cols 1024-1500  size 512 x 476
rows  512-1000  cols    0-512   size 488 x 512
rows  512-1000  cols  512-1024  size 488 x 512
rows  512-1000  cols 1024-1500  size 488 x 476


Let us understand the code above:
- The iterator moves **right** across the image first, then goes **down** to the next row of tiles, just like reading a book.
- The tiles at the right edge and bottom edge are **smaller**, because the image size is not an exact multiple of 512. `min()` makes sure we never go outside the image.
- The iterator only remembers **where it is**. It does not store all the tiles, so it uses almost no memory.

## **Built-in Functions That Return Iterators**

In [6]:
m = map(str.upper, ["digha", "puri"])
z = zip("ab", [1, 2])
e = enumerate("xy")
r = reversed([1, 2, 3])

for x in (m, z, e, r):
    print(type(x).__name__, "->", list(x))

map -> ['DIGHA', 'PURI']
zip -> [('a', 1), ('b', 2)]
enumerate -> [(0, 'x'), (1, 'y')]
list_reverseiterator -> [3, 2, 1]


## **iter() with a Stop Value**
`iter(function, stop_value)` keeps calling the function until it returns the stop value.

In [7]:
import random
random.seed(3)
roll = lambda: random.randint(1, 6)
print("Dice rolls until we get a 6:", list(iter(roll, 6)))

Dice rolls until we get a 6: [2, 5, 5, 2, 3, 5, 4]


In [8]:
# Quick question
# Guess the output of this code cell
# it = iter("abc")
# next(it)
# print(list(it))